In [1]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index
from openai import OpenAI
from rag_helper import RAGBase


load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

### Terminology time ! 

##### Trace, n. 
A **trace** is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.  

##### Span, n. 
A **span** is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, a start and end time, and a set of attributes. For us we will have one span inside the trace, but for agents one trace will have multiple spans.  

##### Attribute, n.
**Attributes** are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.

In [2]:
from db_init_hw import SQLiteSpanExporter

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

#Creates the SDK's central configuration object
provider = TracerProvider() 

# Writes a processor that forwards every finished span to the console exporter one at a time
# FYI: 'Simple' means synchronous and immediate
"""
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
) 
"""
#Writes the results to a database for questions 5 onwards
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

# Registers the provider globally so every call to `trace.get_tracer(...)` returns a tracer backed by it
trace.set_tracer_provider(provider)

# Returns a `Tracer` we use to create spans. 
tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    """
    Subclass of RAGBase that wraps key methods with OpenTelemetry spans
    for observability and tracing.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_call: LLMCallRecord = None

    def search(self, query, num_results=5):
        """Search with tracing span."""
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            
            results = super().search(query, num_results=num_results)
            
            span.set_attribute("results_count", len(results))
            return results

    def llm(self, prompt):
        """LLM call with tracing span."""
        with tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("model", self.model)
            span.set_attribute("prompt_length", len(prompt))
            
            response = super().llm(prompt)
            usage = response.usage

            # Add response attributes if available
            if hasattr(response, 'output_text'):
                span.set_attribute("response_length", len(response.output_text))
                span.set_attribute("input_tokens", usage.input_tokens)
                span.set_attribute("output_tokens", usage.output_tokens)

            
            return response

    def rag(self, query):
        """Full RAG pipeline with tracing span."""
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            span.set_attribute("model", self.model)
            
            result = super().rag(query)
            
            span.set_attribute("result_length", len(result))
            return result

In [4]:
rag_traced = RAGTraced(
    index=index,
    llm_client=client
)

In [6]:
runs = 0 

while runs < 4: 

    query = "How does the agentic loop keep calling the model until it stops?"
    answer = rag_traced.rag(query)
    print(answer)

    print(runs)
    runs += 1

The loop keeps calling the model with a `while True` loop and stops only when the response contains no `function_call` items.

In the code:

- it sends `messages` to the model
- if the model returns function calls, it runs them and appends the tool outputs
- it sets `has_function_calls = True`
- if there are no function calls in that turn, it breaks out of the loop

So the exit condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps looping until the model gives a final message without asking for any more tools.
0
It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tools, appends the results to `messages`, and loops again.
- If there are no function calls, it `break`s out of the loop.

So the stop condition is: **no function calls in the latest response**.
1
It keeps calling the model in a `while True` loop.

Eac

In [7]:
import sqlite3
import pandas as pd 

conn = sqlite3.connect('traces.db')

df = pd.read_sql_query("Select * from spans", conn)

conn.close()

In [8]:
df['start_time_utc'] = pd.to_datetime(df["start_time"])

df['end_time_utc'] = pd.to_datetime(df["end_time"])
df["time_taken"] = df['end_time'] - df["start_time"]


df["time_taken_utc"] = df['end_time_utc'] - df["start_time_utc"]

In [9]:
df.sort_values(by='time_taken_utc', ascending=False)

,name,start_time,end_time,input_tokens,output_tokens,cost,start_time_utc,end_time_utc,time_taken,time_taken_utc
2,rag,1784737524049372000,1784737527043809000,NaN,NaN,None,2026-07-22 16:25:24.049372,2026-07-22 16:25:27.043809,2994437000,0 days 00:00:02.994437
1,llm_call,1784737524069737000,1784737527043241000,7111.0,132.0,None,2026-07-22 16:25:24.069737,2026-07-22 16:25:27.043241,2973504000,0 days 00:00:02.973504
8,rag,1784737528545190000,1784737530577193000,NaN,NaN,None,2026-07-22 16:25:28.545190,2026-07-22 16:25:30.577193,2032003000,0 days 00:00:02.032003
7,llm_call,1784737528547238000,1784737530573364000,7111.0,99.0,None,2026-07-22 16:25:28.547238,2026-07-22 16:25:30.573364,2026126000,0 days 00:00:02.026126
11,rag,1784737530579148000,1784737532152394000,NaN,NaN,None,2026-07-22 16:25:30.579148,2026-07-22 16:25:32.152394,1573246000,0 days 00:00:01.573246
10,llm_call,1784737530585922000,1784737532151007000,7111.0,88.0,None,2026-07-22 16:25:30.585922,2026-07-22 16:25:32.151007,1565085000,0 days 00:00:01.565085
5,rag,1784737527044255000,1784737528544513000,NaN,NaN,None,2026-07-22 16:25:27.044255,2026-07-22 16:25:28.544513,1500258000,0 days 00:00:01.500258
4,llm_call,1784737527045594000,1784737528543583000,7111.0,92.0,None,2026-07-22 16:25:27.045594,2026-07-22 16:25:28.543583,1497989000,0 days 00:00:01.497989
0,search,1784737524049735000,1784737524068018000,NaN,NaN,None,2026-07-22 16:25:24.049735,2026-07-22 16:25:24.068018,18283000,0 days 00:00:00.018283
9,search,1784737530579270000,1784737530583349000,NaN,NaN,None,2026-07-22 16:25:30.579270,2026-07-22 16:25:30.583349,4079000,0 days 00:00:00.004079
